In [0]:

display(dbutils.fs.ls("abfss://bronze@stcommercesrc.dfs.core.windows.net/payments/"))

path,name,size,modificationTime
abfss://bronze@stcommercesrc.dfs.core.windows.net/payments/data_a50ffb8e-0fc1-468b-b0cb-ffd548b125bf_c1fe972f-6ac0-4bea-83c9-0265650e888a.parquet,data_a50ffb8e-0fc1-468b-b0cb-ffd548b125bf_c1fe972f-6ac0-4bea-83c9-0265650e888a.parquet,5614007,1790965080000


In [0]:
df_bronze_payments = spark.read.format('parquet')\
                                .option('header',True)\
                                .option('inferSchema',True)\
                                .load("abfss://bronze@stcommercesrc.dfs.core.windows.net/payments")
display(df_bronze_payments)

payment_id,order_id,payment_method,transaction_id,payment_amount,payment_status,payment_date,refund_amount,created_at,updated_at
1,83811,UPI,TXN0000000001,36867.050000000000000000,SUCCESS,2026-09-20T07:33:32.000Z,0E-18,2026-09-20T07:33:32.000Z,2026-09-24T07:33:32.000Z
2,32099,CREDIT_CARD,TXN0000000002,5916.060000000000000000,SUCCESS,2024-06-21T22:43:23.000Z,0E-18,2024-06-21T22:43:23.000Z,2024-06-22T22:43:23.000Z
3,88697,COD,TXN0000000003,0E-18,PENDING,2024-02-08T20:28:06.000Z,0E-18,2024-02-08T20:28:06.000Z,2024-02-09T20:28:06.000Z
4,77398,NET_BANKING,TXN0000000004,70865.190000000000000000,SUCCESS,2025-03-03T05:50:10.000Z,0E-18,2025-03-03T05:50:10.000Z,2025-03-04T05:50:10.000Z
5,28658,CREDIT_CARD,TXN0000000005,53180.940000000000000000,SUCCESS,2025-01-15T10:13:48.000Z,0E-18,2025-01-15T10:13:48.000Z,2025-01-15T10:13:48.000Z
6,73564,CREDIT_CARD,TXN0000000006,48036.500000000000000000,SUCCESS,2024-12-12T17:53:58.000Z,0E-18,2024-12-12T17:53:58.000Z,2024-12-20T17:53:58.000Z
7,54988,CREDIT_CARD,TXN0000000007,68370.380000000000000000,SUCCESS,2024-08-04T18:21:03.000Z,0E-18,2024-08-04T18:21:03.000Z,2024-08-08T18:21:03.000Z
8,852,CREDIT_CARD,TXN0000000008,90221.310000000000000000,SUCCESS,2024-06-08T05:17:49.000Z,0E-18,2024-06-08T05:17:49.000Z,2024-06-13T05:17:49.000Z
9,36422,CREDIT_CARD,TXN0000000009,86325.740000000000000000,SUCCESS,2026-04-27T04:44:17.000Z,0E-18,2026-04-27T04:44:17.000Z,2026-05-02T04:44:17.000Z
10,13397,UPI,TXN0000000010,68498.680000000000000000,SUCCESS,2024-05-15T01:18:52.000Z,0E-18,2024-05-15T01:18:52.000Z,2024-05-20T01:18:52.000Z


In [0]:
%sql
CREATE TABLE ecommerce.bronze.payments
USING PARQUET
LOCATION 'abfss://bronze@stcommercesrc.dfs.core.windows.net/payments/'

In [0]:
%sql
SELECT COUNT(*) AS duplicate_payments
FROM ecommerce.bronze.payments
GROUP BY payment_id HAVING COUNT(*) > 1

duplicate_payments


In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT payment_id) AS unique_payment_ids,
    COUNT(DISTINCT transaction_id) AS unique_transaction_ids
FROM ecommerce.bronze.payments;

total_records,unique_payment_ids,unique_transaction_ids
100000,100000,99999


In [0]:
%sql

SELECT
    transaction_id,
    COUNT(*) AS occurrence_count
FROM ecommerce.bronze.payments
GROUP BY transaction_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

transaction_id,occurrence_count
TXN0000001001,2


In [0]:
%sql

SELECT *
FROM ecommerce.bronze.payments
WHERE transaction_id = 'TXN0000001001'
ORDER BY payment_id;

payment_id,order_id,payment_method,transaction_id,payment_amount,payment_status,payment_date,refund_amount,created_at,updated_at
1001,65812,WALLET,TXN0000001001,0E-18,PENDING,2025-09-25T18:59:43Z,0E-18,2025-09-25T18:59:43Z,2025-10-04T18:59:43Z
1002,88806,WALLET,TXN0000001001,8942.800000000000000000,SUCCESS,2024-05-24T19:05:27Z,0E-18,2024-05-24T19:05:27Z,2024-05-31T19:05:27Z


In [0]:
%sql

SELECT
    p.order_id,
    COUNT(*) AS payment_count
FROM ecommerce.bronze.payments p
LEFT JOIN ecommerce.silver.orders o
    ON p.order_id = o.order_id
WHERE o.order_id IS NULL
GROUP BY p.order_id
ORDER BY payment_count DESC;

order_id,payment_count


In [0]:
%sql

SELECT COUNT(*) AS invalid_payment_amounts
FROM ecommerce.bronze.payments
WHERE payment_amount IS NULL
   OR payment_amount < 0
   OR refund_amount IS NULL
   OR refund_amount < 0;

invalid_payment_amounts
0


In [0]:
%sql

SELECT
    payment_status,
    COUNT(*) AS payment_count
FROM ecommerce.bronze.payments
GROUP BY payment_status
ORDER BY payment_count DESC;

payment_status,payment_count
SUCCESS,79966
FAILED,8079
PENDING,6966
REFUNDED,4989


In [0]:
%sql

SELECT COUNT(*) AS invalid_payment_dates
FROM ecommerce.bronze.payments
WHERE payment_date IS NULL
   OR created_at IS NULL
   OR updated_at IS NULL
   OR updated_at < created_at;

invalid_payment_dates
0


In [0]:
%sql

SELECT COUNT(*) AS future_payments
FROM ecommerce.bronze.payments
WHERE payment_date > CURRENT_TIMESTAMP();

future_payments
0


In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce.silver.payments
USING DELTA
AS
SELECT
    payment_id,
    order_id,
    INITCAP(TRIM(payment_method)) AS payment_method,
    TRIM(transaction_id) AS transaction_id,
    CAST(payment_amount AS DECIMAL(12,2)) AS payment_amount,
    INITCAP(TRIM(payment_status)) AS payment_status,
    payment_date,
    CAST(refund_amount AS DECIMAL(12,2)) AS refund_amount,
    created_at,
    updated_at
FROM ecommerce.bronze.payments
WHERE payment_id IS NOT NULL
  AND order_id IS NOT NULL
  AND transaction_id IS NOT NULL
  AND TRIM(transaction_id) <> ''
  AND payment_amount IS NOT NULL
  AND payment_amount >= 0
  AND refund_amount IS NOT NULL
  AND refund_amount >= 0
  AND payment_date IS NOT NULL
  AND created_at IS NOT NULL
  AND updated_at IS NOT NULL
  AND updated_at >= created_at
  AND UPPER(TRIM(payment_status)) IN (
      'SUCCESS',
      'FAILED',
      'PENDING',
      'REFUNDED'
  );

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS silver_payment_count
FROM ecommerce.silver.payments;

silver_payment_count
100000


In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce.silver.payment_duplicate_exceptions
USING DELTA
AS
WITH duplicate_transactions AS (
    SELECT
        transaction_id
    FROM ecommerce.silver.payments
    GROUP BY transaction_id
    HAVING COUNT(*) > 1
)
SELECT
    p.payment_id,
    p.order_id,
    p.payment_method,
    p.transaction_id,
    p.payment_amount,
    p.payment_status,
    p.payment_date,
    p.refund_amount,
    p.created_at,
    p.updated_at,
    'DUPLICATE_TRANSACTION_ID' AS exception_reason,
    CURRENT_TIMESTAMP() AS exception_detected_at
FROM ecommerce.silver.payments p
INNER JOIN duplicate_transactions d
    ON p.transaction_id = d.transaction_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    transaction_id,
    COUNT(*) AS occurrence_count
FROM ecommerce.silver.payment_duplicate_exceptions
GROUP BY transaction_id;

transaction_id,occurrence_count
TXN0000001001,2


In [0]:
%sql

SELECT *
FROM ecommerce.silver.payment_duplicate_exceptions
ORDER BY payment_id;

payment_id,order_id,payment_method,transaction_id,payment_amount,payment_status,payment_date,refund_amount,created_at,updated_at,exception_reason,exception_detected_at
1001,65812,Wallet,TXN0000001001,0.00,Pending,2025-09-25T18:59:43Z,0.00,2025-09-25T18:59:43Z,2025-10-04T18:59:43Z,DUPLICATE_TRANSACTION_ID,2026-10-03T14:11:22.089177Z
1002,88806,Wallet,TXN0000001001,8942.80,Success,2024-05-24T19:05:27Z,0.00,2024-05-24T19:05:27Z,2024-05-31T19:05:27Z,DUPLICATE_TRANSACTION_ID,2026-10-03T14:11:22.089177Z


In [0]:
%sql

CREATE TABLE ecommerce.silver.payments_external
USING DELTA
LOCATION 'abfss://silver@stcommercesrc.dfs.core.windows.net/payments/'
AS
SELECT *
FROM ecommerce.silver.payments;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS external_payment_count
FROM ecommerce.silver.payments_external;

external_payment_count
100000


In [0]:
%sql

DESCRIBE HISTORY ecommerce.silver.payments_external;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-03T14:12:15Z,148417918535894,saireddy12315_outlook.com#ext#@saireddy12315outlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1220552743077210),1003-052435-xepp1xgc,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 100000, numOutputBytes -> 4827155)",null,Databricks-Runtime/17.3.x-photon-scala2.13


In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce.silver.payment_quality_metrics
USING DELTA
AS
SELECT
    CURRENT_TIMESTAMP() AS validation_timestamp,

    (SELECT COUNT(*)
     FROM ecommerce.bronze.payments) AS bronze_records,

    (SELECT COUNT(*)
     FROM ecommerce.silver.payments) AS silver_records,

    (SELECT COUNT(*)
     FROM ecommerce.silver.payment_duplicate_exceptions) AS duplicate_exception_records,

    ROUND(
        (
            (SELECT COUNT(*)
             FROM ecommerce.silver.payments)
            * 100.0
        )
        /
        (SELECT COUNT(*)
         FROM ecommerce.bronze.payments),
        2
    ) AS transformation_quality_percentage;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM ecommerce.silver.payment_quality_metrics;

validation_timestamp,bronze_records,silver_records,duplicate_exception_records,transformation_quality_percentage
2026-10-03T14:13:11.833687Z,100000,100000,2,100.00


In [0]:
%sql
SELECT * FROM delta.`abfss://silver@stcommercesrc.dfs.core.windows.net/payments/`

payment_id,order_id,payment_method,transaction_id,payment_amount,payment_status,payment_date,refund_amount,created_at,updated_at
1,83811,Upi,TXN0000000001,36867.05,Success,2026-09-20T07:33:32Z,0.00,2026-09-20T07:33:32Z,2026-09-24T07:33:32Z
2,32099,Credit_card,TXN0000000002,5916.06,Success,2024-06-21T22:43:23Z,0.00,2024-06-21T22:43:23Z,2024-06-22T22:43:23Z
3,88697,Cod,TXN0000000003,0.00,Pending,2024-02-08T20:28:06Z,0.00,2024-02-08T20:28:06Z,2024-02-09T20:28:06Z
4,77398,Net_banking,TXN0000000004,70865.19,Success,2025-03-03T05:50:10Z,0.00,2025-03-03T05:50:10Z,2025-03-04T05:50:10Z
5,28658,Credit_card,TXN0000000005,53180.94,Success,2025-01-15T10:13:48Z,0.00,2025-01-15T10:13:48Z,2025-01-15T10:13:48Z
6,73564,Credit_card,TXN0000000006,48036.50,Success,2024-12-12T17:53:58Z,0.00,2024-12-12T17:53:58Z,2024-12-20T17:53:58Z
7,54988,Credit_card,TXN0000000007,68370.38,Success,2024-08-04T18:21:03Z,0.00,2024-08-04T18:21:03Z,2024-08-08T18:21:03Z
8,852,Credit_card,TXN0000000008,90221.31,Success,2024-06-08T05:17:49Z,0.00,2024-06-08T05:17:49Z,2024-06-13T05:17:49Z
9,36422,Credit_card,TXN0000000009,86325.74,Success,2026-04-27T04:44:17Z,0.00,2026-04-27T04:44:17Z,2026-05-02T04:44:17Z
10,13397,Upi,TXN0000000010,68498.68,Success,2024-05-15T01:18:52Z,0.00,2024-05-15T01:18:52Z,2024-05-20T01:18:52Z


In [0]:
%sql
SELECT COUNT(*) FROM ecommerce.silver.payments
WHERE payment_status = 'Success'

count(1)
79966
